# Lunes 

<b> Estadística </b>

¿Shrinkage ayuda realmente a estimar alpha?
Ahora vamos a probarlo con simulación.
El verdadero proceso es:

```python
import numpy as np

np.random.seed(117)

true_mu = 0.0003     # 3 bps/día
sigma = 0.008        # 80 bps/día
n = 126
M = 5000

prior_mu = 0.0
prior_sd = 0.0004    # 4 bps/día
```


Simula $M = 5000$ research samples de $126$ días.

Para cada experimento calcula:

$$\hat{\mu}_j = \bar{r}_j.$$

Después calcula el posterior:

$$\mu_{post,j} = \frac{\mu_0 / \tau^2 + \hat{\mu}_j / SE^2}{1 / \tau^2 + 1 / SE^2}$$

donde:

$$SE = \frac{\sigma}{\sqrt{n}}.$$

Compara los dos estimadores:

$$\hat{\mu}$$

y

$$\mu_{post}.$$

Reporta

| Métrica | Sample mean | Posterior mean |
|---|---:|---:|
| Bias | | |
| Std estimator | | |
| RMSE | | |
| 5% quantile | | |
| 95% quantile | | |


Después responde:

> ¿Cómo puede el posterior introducir algo de bias y aun así tener menor RMSE?

Quiero que conectes:

$$RMSE^2 = Bias^2 + Variance.$$

Ésta es la idea fundamental detrás de shrinkage.

Ahora la pregunta más importante:


> Si el verdadero alpha fuera enorme, ¿seguiría siendo conveniente aplicar el mismo prior agresivamente?

No respondas simplemente "no".

Explica qué ocurre cuando el prior está mal calibrado y por qué shrinkage tampoco es magia.


### <b> Quant Research + Quant Desk </b>

#### De alpha estimado a alpha utilizable

Tu Research Engine produce:

```text
strategy_id       REV_024

sample_alpha      +7.0 bps/day
alpha_SE          4.0 bps/day

sample_sharpe     1.28
OOS_days          252
```

Tu capa estadística devuelve:

```text
posterior_alpha   +3.1 bps/day
posterior_SD      2.4 bps/day
```

Ahora imagina que el Portfolio Engine recibe solamente:

```text
expected_alpha = 7.0 bps/day
```

y genera una posición de:

\[28\%.\]

Pero si recibe:

```text
expected_alpha = 3.1 bps/day
```

genera:

$$12\%.$$

Finalmente Risk aplica:

$$w_{\max} = 10\%.$$

Por tanto:

$$28\% \rightarrow 12\% \rightarrow 10\%.$$

Tu tarea no es decidir cuál fórmula exacta de sizing usar todavía.

Define conceptualmente estas capas:


Observed Alpha
      ↓
Statistical Belief Layer
      ↓
Portfolio Sizing
      ↓
Risk Cap.

Ahora responde:

> ¿Por qué no debería Research entregar simplemente un único número llamado `expected_return` ?

Diseña un objeto mínimo:

```text
sample_alpha

sampling_error
prior_alpha
prior_uncertainty

posterior_alpha
posterior_uncertainty

prob_alpha_positive

research_status
```

Después decide qué debería significar:

$$P(\mu > 0 \mid data) = 0.62$$

frente a:

$$P(\mu > 0 \mid data) = 0.97.$$

No conviertas todavía esos números en thresholds automáticos.

Quiero que discutas algo más profundo:

> ¿Debe una estrategia con posterior alpha menor recibir siempre menos capital?

No necesariamente.

También importan:

$$\Sigma,\quad \text{correlaciones},\quad \text{liquidez},\quad \text{costes},\quad \text{diversificación}.$$

Termina explicando por qué:

$$\text{belief about alpha}$$

y

$$\text{optimal portfolio allocation}$$

son problemas distintos aunque estén conectados.



# MARTES 


###  Matemáticas + factor models · A mano

#### ¿Tu alpha es realmente alpha o exposición a factores?

Tienes tres activos y dos factores.

$$w = \begin{bmatrix} 0.50 \\ 0.30 \\ 0.20 \end{bmatrix}$$

Sus exposiciones factoriales son:

$$F = \begin{bmatrix} 1.2 & 0.3 \\ 0.8 & -0.2 \\ 0.1 & 1.1 \end{bmatrix}$$

donde las columnas representan:

$$f_1 = \text{Market}$$

$$f_2 = \text{Momentum}.$$

Calcula primero la exposición del portfolio:

$$\beta_P = F^{\top} w.$$

Ahora Research afirma que el portfolio tiene un retorno esperado:

$$E[r_P] = 8\%.$$

Pero estima:

$$E[f_1] = 5\%$$

$$E[f_2] = 3\%.$$

Calcula la parte del retorno explicada aproximadamente por factores:

$$E[r_P]_{\text{factor}} = \beta_{P,1}E[f_1] + \beta_{P,2}E[f_2].$$

Después define el residual:

$$\alpha_P = E[r_P] - E[r_P]_{\text{factor}}.$$

La parte importante no es el número.

Responde:

> Si el portfolio gana 8%, ¿por qué sería incorrecto llamar automáticamente 8% "alpha"?

Después explica la diferencia conceptual entre:

$$\text{return}$$

$$\text{factor exposure}$$

$$\text{residual alpha}.$$

Y finalmente responde:

> Si neutralizas Market y Momentum, ¿esperas conservar necesariamente el mismo retorno esperado?

Tu respuesta debe reconocer que parte del “alpha” original puede desaparecer porque nunca fue alpha independiente.



###  Portfolio/Risk + Entropy Pooling · A mano

#### Una view no es una verdad

Tu prior de escenarios para el retorno mensual de un activo es:

| Escenario | Retorno | Probabilidad prior |
| :--- | :---: | :---: |
| Crash | -8% | 0.20 |
| Flat | 0% | 0.50 |
| Rally | +6% | 0.30 |

Calcula primero:

$$E[r]_{\text{prior}}.$$

Ahora Research produce una view:

> "Creo que el retorno esperado debería ser aproximadamente +2.5%."

No quiero que cambies directamente los retornos de los escenarios.

Tu trabajo es razonar cómo Entropy Pooling trataría el problema:

$$\text{mantener escenarios}$$

pero modificar:

$$\text{probabilidades}.$$

Responde:

¿Hacia qué escenarios debería desplazarse probabilidad?

¿De cuál debería quitarse?

¿Por qué no queremos alterar más de lo necesario la distribución prior?


Relaciona eso con la idea:

$$\min_{p} \sum_{i} p_i \log \left( \frac{p_i}{q_i} \right)$$

donde $q_i$ son probabilidades prior y $p_i$ las posteriores.

No necesitas resolver la optimización.

Quiero que expliques qué significa económicamente minimizar esa distancia.

Después aparece incertidumbre en Research.


Caso A:

$$\text{confidence} = 95\%.$$

Caso B:

$$\text{confidence} = 55\%.$$

¿Qué debería ocurrir con la distancia entre:

$$p \quad \text{y} \quad q$$

en cada caso?

La intuición que quiero:

$$\text{más confianza} \implies \text{mayor desplazamiento permitido desde el prior}$$

y

$$\text{menos confianza} \implies \text{posterior más cercano al prior}.$$

Finalmente conecta esto con lo de ayer:

$$\hat{\mu} \rightarrow \text{posterior belief}$$

y hoy:

$$\text{prior distribution} \rightarrow \text{view-conditioned distribution}.$$


¿Son exactamente lo mismo? No.

Pero ambos comparten una idea:

$$\text{evidence should update beliefs, not replace them blindly}.$$


# MIERCOLES

**probabilidad**

Impacto vs riesgo de esperar

Vamos a construir un modelo deliberadamente simple.

Supón que tienes que ejecutar:

$ Q = 100,000 $ 

acciones durante 10 intervalos.

```python
import numpy as np

Q = 100_000
N = 10

sigma = 0.0015
eta = 2e-7
```

Para cada intervalo $t$, ejecutas:

$q_t$.

Modelo simplificado de coste:

$C_{impact} = \eta \sum_{t} q_t^2$

y riesgo de inventario:

$C_{risk} = \sigma^2 \sum_{t} X_t^2$

donde $X_t$ es el inventario restante antes de cada intervalo.

Construye tres schedules:

**TWAP**

$q_t = \frac{Q}{N}.$

**Front-loaded**

Ejecuta:

$20\%, 15\%, 15\%, 10\%, 10\%, 10\%, 7\%, 5\%, 5\%, 3\%.$

Back-loaded
Usa exactamente esos pesos en orden inverso.

| Schedule | Impact cost | Inventory-risk proxy | Total proxy |
|---|---:|---:|---:|
| TWAP | | | |
| Front-loaded | | | |
| Back-loaded | | | |

Define:

$Total = C_{impact} + \lambda C_{risk}$

con:

$\lambda = 10^5.$

No busco realismo en las unidades. Es un experimento conceptual.

Después responde:


> ¿Por qué el front-loaded puede reducir riesgo de inventario pero aumentar impacto?

Y:

> ¿Por qué el schedule óptimo depende de $\lambda$?

Interpreta $\lambda$ como aversión al riesgo de esperar.

Finalmente razona:

Si la señal tiene half-life muy corta, ¿esperarías un $\lambda$ económico más alto o más bajo?

No lo reduzcas a una fórmula. Explícalo desde la vida útil del alpha.

**3. 20 min — Research reading + Quant Desk**

Una estrategia puede tener alpha y aun así no ser ejecutable

Lee este resultado ficticio como si fuera un paper interno:

> Una señal intradía sobre acciones líquidas obtiene un alpha bruto medio de 8.4 bps por trade. El turnover diario es 180%. El backtest usa mid-price como precio de entrada. El signal half-life es de 12 minutos. Cuando se incorpora un modelo simple de spread, el alpha cae a 5.2 bps. Con impacto estimado bajo una participación del 15% del volumen, el alpha neto cae a 1.1 bps.

Tu tarea es hacer una lectura crítica, no repetir el resultado.

Responde estas cinco preguntas:

1. ¿Qué supuesto del backtest original era demasiado optimista?
2. ¿Por qué una señal con half-life de 12 minutos impone una restricción fuerte a la ejecución?
3. ¿Por qué no basta con decir:

$$\text{alpha bruto} > 0$$

para decidir que la estrategia es deployable?
4. ¿Qué cambiaría si para reducir impacto bajaras participation rate de:

$15\% \rightarrow 5\%$?

5. ¿Qué podría ocurrir si, al ejecutar más despacio, consumes una fracción relevante de la vida del alpha?

Ahora llévalo a tu Quant Desk.

Diseña un Execution Gate mínimo:


